# 선형회귀 모델 작성, 예측, 평가

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
%matplotlib inline

In [8]:
charges_df = pd.read_csv('./머신러닝 보고서/data1/premium.csv')
charges_df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [9]:
# 'bmi' 컬럼 결측값에 평균값으로 채우기
charges_df['bmi'] = charges_df['bmi'].fillna(charges_df['bmi'].mean())

+ 'sex', 'smoker', 'region' 컬럼 레이블 인코딩

In [10]:
from sklearn.preprocessing import LabelEncoder

# 레이블 인코딩을 적용할 컬럼 리스트
columns_label = ['sex', 'smoker', 'region']

# 복사본 생성
charges_df1 = charges_df.copy()

# 각 컬럼에 대해 LabelEncoder 적용
for col in columns_label:
    if col in charges_df1.columns: # 컬럼 존재 여부 확인 (방어 코드)
        le = LabelEncoder() # 각 컬럼마다 새로운 LabelEncoder 객체를 생성
        charges_df1[col] = le.fit_transform(charges_df1[col])

        print(f"\n--- '{col}' 컬럼 인코딩 후 매핑 정보 ---")
        # 인코딩된 값과 원래 문자열 값의 매핑 확인
        for i, class_name in enumerate(le.classes_):
            print(f"  {class_name}: {i}")
    else:
        print(f"경고: '{col}' 컬럼이 데이터프레임에 존재하지 않습니다.")

print("\n--- 레이블 인코딩 후 데이터 head() ---")
print(charges_df1.head())

print("\n--- 레이블 인코딩 후 데이터 컬럼 Dtypes 확인 ---")
print(charges_df1.dtypes)


--- 'sex' 컬럼 인코딩 후 매핑 정보 ---
  female: 0
  male: 1

--- 'smoker' 컬럼 인코딩 후 매핑 정보 ---
  no: 0
  yes: 1

--- 'region' 컬럼 인코딩 후 매핑 정보 ---
  northeast: 0
  northwest: 1
  southeast: 2
  southwest: 3

--- 레이블 인코딩 후 데이터 head() ---
   age  sex     bmi  children  smoker  region      charges
0   19    0  27.900         0       1       3  16884.92400
1   18    1  33.770         1       0       2   1725.55230
2   28    1  33.000         3       0       2   4449.46200
3   33    1  22.705         0       0       1  21984.47061
4   32    1  28.880         0       0       1   3866.85520

--- 레이블 인코딩 후 데이터 컬럼 Dtypes 확인 ---
age           int64
sex           int64
bmi         float64
children      int64
smoker        int64
region        int64
charges     float64
dtype: object


In [11]:
X = charges_df1.drop('charges', axis=1).values   # 독립변수
y = charges_df1['charges'].values   # 종속변수

## 모델 만들기

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=156)
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)
y_pred[:3]

array([14474.70246359, -1367.94174448, 11182.0795591 ])

In [13]:
# 평가
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
rmse, mse

(np.float64(5892.287122021437), np.float64(34719047.52833967))

+ RMSE는 약 5892.29이다

In [14]:
# 결정계수
r2_score(y_test, y_pred)

np.float64(0.7314050294401666)

+ 결정계수 r2스코어는 약 0.731이다

In [15]:
# 회귀식
# w0, w1
lr.intercept_, lr.coef_

(np.float64(-12749.427561095086),
 array([  257.5385329 ,  -339.97677884,   369.66261678,   471.40493778,
        23624.46109983,  -375.59873801]))

In [16]:
np.round(lr.intercept_, 1), np.round(lr.coef_, 1)

(np.float64(-12749.4),
 array([  257.5,  -340. ,   369.7,   471.4, 23624.5,  -375.6]))

In [17]:
pd.Series(data=np.round(lr.coef_, 1), index=charges_df1.drop('charges', axis=1).columns).sort_values(ascending=False)

smoker      23624.5
children      471.4
bmi           369.7
age           257.5
sex          -340.0
region       -375.6
dtype: float64

# 랜덤포레스트회귀 모델 작성, 예측, 평가

In [18]:
from sklearn.ensemble import RandomForestRegressor

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=156)
rf = RandomForestRegressor()
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
y_pred[:3]

array([16571.3143064,  2153.5101043,  8839.0224714])

In [20]:
# 평가
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
rmse, mse

(np.float64(4734.607819666283), np.float64(22416511.20604511))

+ RMSE는 약 4810.36이다

In [21]:
# 결정계수
r2_score(y_test, y_pred)

np.float64(0.8265804336214233)

+ 결정계수 r2스코어는 약 0.821이다

In [22]:
# 피처 중요도 확인
feature_importances = rf.feature_importances_
feature_names = charges_df1.drop('charges', axis=1).columns

# 각 피처의 이름과 중요도를 매칭하여 출력
for feature, importance in zip(feature_names, feature_importances):
    print(f"{feature}: {importance:.4f}")

age: 0.1266
sex: 0.0056
bmi: 0.2131
children: 0.0184
smoker: 0.6211
region: 0.0152


## 교차 검증

In [23]:
# 선형회귀모델 교차 검증
from sklearn.model_selection import cross_val_score
neg_mse_score = cross_val_score(lr, X, y, scoring='neg_mean_squared_error', cv=5)
neg_mse_score

array([-37353966.14780101, -38018280.71475136, -32981193.39000173,
       -39560881.14778336, -37174240.90789752])

In [24]:
# MSE, RMSE
RMSE = np.sqrt(neg_mse_score * -1)
np.mean(RMSE), RMSE

(np.float64(6081.484710559383),
 array([6111.78911186, 6165.89658645, 5742.92550796, 6289.74412419,
        6097.06822234]))

+ 교차검증 후 RMSE는 약 6081.48이다

In [25]:
# R2
r2_scores = cross_val_score(lr, X, y, scoring='r2', cv=5)
r2_scores, np.mean(r2_scores)

(array([0.75962321, 0.70729102, 0.77528105, 0.73350581, 0.7552539 ]),
 np.float64(0.7461909971637163))

+ 교차검증 후 R2스코어는 약 0.75이다

In [26]:
# 랜덤포레스트모델 교차 검증
from sklearn.model_selection import cross_val_score
neg_mse_score = cross_val_score(rf, X, y, scoring='neg_mean_squared_error', cv=5)
neg_mse_score

array([-22761133.94476196, -29765747.1678093 , -21646284.11855507,
       -25896138.29597462, -22801417.6547773 ])

In [27]:
# MSE, RMSE
RMSE = np.sqrt(neg_mse_score * -1)
np.mean(RMSE), RMSE

(np.float64(4948.625399801984),
 array([4770.86301886, 5455.79940685, 4652.55672921, 5088.8248443 ,
        4775.08299978]))

+ 교차검증 후 RMSE는 약 4951.42이다

In [28]:
# R2
r2_scores = cross_val_score(rf, X, y, scoring='r2', cv=5)
r2_scores, np.mean(r2_scores)

(array([0.85040278, 0.76668178, 0.84698878, 0.83005296, 0.84944785]),
 np.float64(0.8287148298479685))

+ 교차검증 후 R2스코어는 약 0.83이다

In [29]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
import numpy as np

In [30]:
# 2차~5차 다항회귀 시뮬레이션 (LineareRegressor)
results = []
best_r2=-np.inf
best_model = None
best_pred = None


for degree in range(1,5):
    model_poly = Pipeline([
        ('poly', PolynomialFeatures(degree=degree, include_bias=False)),
        ('linear', LinearRegression())]
    )
    model_poly.fit(X_train, y_train)
    pred_poly = model_poly.predict(X_test)
    mse = mean_squared_error(y_test, pred_poly)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, pred_poly)
    results.append({'degree':degree, 'MSE':mse, 'RMSE':rmse, 'R2':r2})
    if r2 > best_r2:
        best_r2 = r2
        best_model = model_poly
        best_pred = pred_poly
    
pd.DataFrame(results)

,degree,MSE,RMSE,R2
0,1,3.471905e+07,5892.287122,0.731405
1,2,2.174138e+07,4662.765769,0.831803
2,3,2.276061e+07,4770.808396,0.823918
3,4,2.992843e+07,5470.687872,0.768466


+ 결과를보니 R2스코어, RMSE가 2차에서 가장 좋게 나타난다

In [46]:
# 2차~5차 다항회귀 시뮬레이션 (RandomForest)
results = []
best_r2=-np.inf
best_model = None
best_pred = None

for degree in range(1,6):
    model_poly = Pipeline([
        ('poly', PolynomialFeatures(degree=degree, include_bias=False)),
        ('RandomForest', RandomForestRegressor())]
    )
    model_poly.fit(X_train, y_train)
    pred_poly = model_poly.predict(X_test)
    mse = mean_squared_error(y_test, pred_poly)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, pred_poly)
    results.append({'degree':degree, 'MSE':mse, 'RMSE':rmse, 'R2':r2})
    if r2 > best_r2:
        best_r2 = r2
        best_model = model_poly
        best_pred = pred_poly
    
pd.DataFrame(results)

,degree,MSE,RMSE,R2
0,1,2.321464e+07,4818.157221,0.820406
1,2,2.358020e+07,4855.944363,0.817578
2,3,2.278692e+07,4773.564753,0.823715
3,4,2.278028e+07,4772.869640,0.823766
4,5,2.323382e+07,4820.146793,0.820258


+ 결과를보니 R2스코어, RMSE가 4차에서 가장 좋게 나타난다

# 규제 회귀 모델

## Ridge

In [45]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet

In [33]:
ridge = Ridge(alpha=1.0)   # 규제 강도
ridge.fit(X_train, y_train)
pred_ridge = ridge.predict(X_test)

mse = mean_squared_error(y_test, pred_ridge)
r2  = r2_score(y_test, pred_ridge)
mse, r2

(np.float64(34752916.781278014), np.float64(0.7311430086866182))

In [35]:
from sklearn.linear_model import RidgeCV, LassoCV
alphas = [0.001, 0.01,0.1, 1, 10, 100]
ridge_cv = RidgeCV(alphas=alphas, cv=5)
ridge_cv.fit(X_train, y_train)
ridge_preds = ridge_cv.predict(X_test)
ridge_mse = mean_squared_error(y_test, ridge_preds)
ridge_r2 = r2_score(y_test, ridge_preds)

print(f'ridge cv mse : {ridge_mse}, r2 : {ridge_r2}')

ridge cv mse : 34752916.781278014, r2 : 0.7311430086866182


In [36]:
ridge_cv.alpha_   # 최적의 알파값

np.float64(1.0)

In [40]:
ridge_cv.coef_

array([  257.5323598 ,  -329.82291803,   369.53011089,   470.20365131,
       23477.8280124 ,  -376.56473999])

## Lasso

In [37]:
lasso = Lasso(alpha=1)   # 규제 강도
lasso.fit(X_train, y_train)
pred_lasso = lasso.predict(X_test)

mse = mean_squared_error(y_test, pred_lasso)
r2 = r2_score(y_test, pred_lasso)
mse, r2

(np.float64(34718257.739815466), np.float64(0.7314111394357025))

In [38]:
from sklearn.linear_model import RidgeCV, LassoCV
alphas = [0.001, 0.01,0.1, 1, 10, 100]
lasso_cv = LassoCV(alphas=alphas, cv=5)
lasso_cv.fit(X_train, y_train)
lasso_preds = lasso_cv.predict(X_test)
lasso_mse = mean_squared_error(y_test, lasso_preds)
lasso_r2 = r2_score(y_test, lasso_preds)
print(f'lasso cv mse : {lasso_mse}, r2 : {lasso_r2}')

lasso cv mse : 34771340.208292864, r2 : 0.7310004805878094


In [39]:
lasso_cv.alpha_   # 최적의 알파값

np.float64(100.0)

In [41]:
lasso_cv.coef_

array([  257.59913589,    -0.        ,   362.54010593,   397.49245805,
       23015.59104426,  -292.03614646])

## 엘라스틱넷

In [42]:
enet = ElasticNet(alpha=0.1, l1_ratio=0.5)
enet.fit(X_train, y_train)

ElasticNet(alpha=0.1)

In [43]:
pred_enet = enet.predict(X_test)
print("엘라스틱넷 회귀")
print(f"MSE : {mean_squared_error(y_test, pred_enet)}")
print(f"R2 : {r2_score(y_test, pred_enet)}")

엘라스틱넷 회귀
MSE : 39905776.36954846
R2 : 0.6912792374157983


# 모델 성능 비교

In [44]:
results = pd.DataFrame({
    '모델':['다항회귀', '릿지회귀', '라쏘회귀', '엘라스틱넷회귀'],
    'MSE' : [mean_squared_error(y_test, pred_poly),
             mean_squared_error(y_test, pred_ridge),
             mean_squared_error(y_test, pred_lasso),
             mean_squared_error(y_test, pred_enet)],
    'R2' : [r2_score(y_test, best_pred),
            r2_score(y_test, pred_ridge),
            r2_score(y_test, pred_lasso),
            r2_score(y_test, pred_enet)]
})
results

,모델,MSE,R2
0,다항회귀,2.365356e+07,0.831803
1,릿지회귀,3.475292e+07,0.731143
2,라쏘회귀,3.471826e+07,0.731411
3,엘라스틱넷회귀,3.990578e+07,0.691279
